# 🚀 PromptForge  
### Batch Prompt Processing Toolkit

PromptForge adalah tool untuk membersihkan, mengelompokkan, dan mengekspor prompt secara otomatis berdasarkan **aspect ratio**.

Tool ini dirancang untuk mempercepat workflow batch generation pada berbagai AI image generator.

## ============================================
## 1. INSTALL DEPENDENCIES
## ============================================
(Turn on only in first time run)

In [5]:
# !pip install ipywidgets

## ============================================
## 2. INTERFACE COMPONENTS
## ============================================

In [6]:
import re
from collections import defaultdict
from google.colab import files
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display

# INPUT MODE
mode_selector = widgets.RadioButtons(
    options=["Paste Text", "Upload File"],
    description="Input Mode:",
)
# TEXT INPUT
text_input = widgets.Textarea(
    placeholder="Paste prompt di sini...",
    layout=widgets.Layout(
        width="100%",
        height="300px"
    )
)
# FILE UPLOAD
file_upload = widgets.FileUpload(
    accept=".txt",
    multiple=False
)
# OPTIONS
remove_duplicate = widgets.Checkbox(
    value=True,
    description="Remove duplicate prompts"
)
add_suffix = widgets.Checkbox(
    value=False,
    description="Auto add suffix"
)
suffix_text = widgets.Text(
    value=", ultra detailed, cinematic lighting, 8k",
    description="Suffix:",
    disabled=True
)
separate_blank = widgets.Checkbox(
    value=False,
    description="Separate with blank space"
)
auto_download = widgets.Checkbox(
    value=True,
    description="Auto download files"
)
auto_zip = widgets.Checkbox(
    value=False,
    description="Auto zip files"
)
# BUTTON
process_button = widgets.Button(
    description="PROCESS PROMPTS",
    button_style="success"
)
# OUTPUT
status_output = widgets.Output()
# CONTAINER
input_container = widgets.VBox()

## ============================================
## 3. UI CONTROLLERS
## ============================================

In [7]:
# @title
def update_input_visibility(change):

    if mode_selector.value == "Paste Text":

        input_container.children = [text_input]

    else:

        input_container.children = [file_upload]


mode_selector.observe(
    update_input_visibility,
    names="value"
)


update_input_visibility(None)

# ENABLE / DISABLE SUFFIX
def toggle_suffix(change):

    suffix_text.disabled = not add_suffix.value


add_suffix.observe(
    toggle_suffix,
    names="value"
)


display(
    mode_selector,
    input_container,
    remove_duplicate,
    add_suffix,
    suffix_text,
    separate_blank,
    auto_download,
    auto_zip,
    process_button,
    status_output
)

RadioButtons(description='Input Mode:', options=('Paste Text', 'Upload File'), value='Paste Text')

Checkbox(value=True, description='Remove duplicate prompts')

Checkbox(value=False, description='Auto add suffix')

Text(value=', ultra detailed, cinematic lighting, 8k', description='Suffix:', disabled=True)

Checkbox(value=False, description='Separate with blank space')

Checkbox(value=True, description='Auto download files')

Checkbox(value=False, description='Auto zip files')

Button(button_style='success', description='PROCESS PROMPTS', style=ButtonStyle())

Output()

Export complete.


## ============================================
## 4. CORE PROCESSING ENGINE
## ============================================

In [8]:
# @title
import re
from collections import defaultdict
from google.colab import files
from datetime import datetime
import os
import zipfile
import ipywidgets as widgets
from IPython.display import display

def clean_prompt(text):

    text = re.sub(r"\s+", " ", text)

    return text.strip()

def process_prompts(b):
    process_button.disabled = True # Disable button at the start
    try:
        with status_output:

            status_output.clear_output()

            # =========================
            # GET RAW TEXT
            # =========================

            if mode_selector.value == "Paste Text":

                raw_text = text_input.value

            else:

                if len(file_upload.value) == 0:

                    print("Upload file dulu.")
                    return

                uploaded_file = list(
                    file_upload.value.values()
                )[0]

                raw_text = uploaded_file[
                    "content"
                ].decode("utf-8")


            ratio_prompt_dict = defaultdict(list)

            blocks = re.split(
                r"---\s*Prompt\s*\d+.*?---",
                raw_text
            )


            for block in blocks:

                ratio_match = re.search(
                    r"Aspect Ratio:\s*(\d+:\d+)",
                    block
                )

                prompt_match = re.search(
                    r"Prompt:\s*(.*)",
                    block,
                    re.DOTALL
                )

                if ratio_match and prompt_match:

                    ratio = ratio_match.group(1)

                    prompt_text = prompt_match.group(1)

                    prompt_text = clean_prompt(
                        prompt_text
                    )


                    if add_suffix.value:

                        prompt_text += suffix_text.value


                    ratio_prompt_dict[
                        ratio
                    ].append(prompt_text)


            # =========================
            # REMOVE DUPLICATE
            # =========================

            if remove_duplicate.value:

                for r in ratio_prompt_dict:

                    ratio_prompt_dict[r] = list(
                        dict.fromkeys(
                            ratio_prompt_dict[r]
                        )
                    )


            # =========================
            # TIMESTAMP AND BATCH FOLDER
            # =========================

            timestamp = datetime.now().strftime(
                "%Y%m%d_%H%M%S"
            )

            batch_folder_name = f"Batch_{timestamp}"
            os.makedirs(batch_folder_name, exist_ok=True)


            print("\nSummary:\n")
            print("Detected aspect ratios:", list(ratio_prompt_dict.keys()))
            print("-" * 30)

            total_prompts_count = 0
            for r in ratio_prompt_dict:

                count = len(ratio_prompt_dict[r])
                print(
                    f"{r} → {count} prompts"
                )
                total_prompts_count += count

            print("-" * 30)
            print(f"Total prompts: {total_prompts_count}\n")

            print(f"Exporting files to folder: {batch_folder_name}/\n")

            generated_files = [] # To store names of generated files for zipping

            # =========================
            # EXPORT FILES
            # =========================

            for ratio, prompts in ratio_prompt_dict.items():

                safe_ratio = ratio.replace(":", "x")

                file_name = (
                    f"{batch_folder_name}/BatchPrompt_{safe_ratio}_{timestamp}.txt"
                )

                with open(
                    file_name,
                    "w",
                    encoding="utf-8"
                ) as f:

                    for p in prompts:

                        if separate_blank.value:

                            f.write(p + "\n\n")

                        else:

                            f.write(p + "\n")

                generated_files.append(file_name)

                # If auto_zip is not selected, download individual files
                if auto_download.value and not auto_zip.value:
                    files.download(file_name)

            # =========================
            # ZIP FILES (if selected)
            # =========================
            if auto_zip.value and generated_files:
                zip_file_path = f"{batch_folder_name}.zip"
                with zipfile.ZipFile(zip_file_path, 'w', zipfile.ZIP_DEFLATED) as zf:
                    for f_name in generated_files:
                        zf.write(f_name, os.path.basename(f_name)) # Add file to zip with just its name
                print(f"Created zip file: {zip_file_path}")

                if auto_download.value:
                    files.download(zip_file_path)
                    print("Zip file downloaded.")


    finally:
        print("Export complete.")
        process_button.disabled = False # Re-enable button in finally block

## ============================================
## 5. EVENT BINDING
## ============================================

In [9]:
# Remove previous handlers (IMPORTANT)
try:
    process_button._click_handlers.callbacks.clear()
except:
    pass

# Bind handler
process_button.on_click(process_prompts)

## 📜 License

This project is released under the MIT License.

You are free to:

- Use
- Modify
- Distribute

With proper attribution.